In [37]:
# import pandas as pd

# # Read TSV files
# types = ["corn", "Ghana", "Liberia", "Mali", "multispectral", "Niger", "Nigeria", "nitrogendilution", "nyield"]
 
# for type in types:
#     df1 = pd.read_csv(f'/AG/App/FlaskApp/DATA/Ag-Data/{type}/final_merged_output.csv', sep='\t')
#     df2 = pd.read_csv(f'/AG/App/FlaskApp/DATA/New_Filtered-Ag/data/{type}-final.csv', sep='\t')
#     classification_results1 = pd.read_csv(f'/AG/App/FlaskApp/DATA/Ag-Data/{type}/final_with_classifications.csv', sep='\t')

#     # Merge classification results with second file based on matching columns
#     merge_cols = ['title','abstract' , 'doi', 'authors', 'count']
    
#     df2_with_class = df2.merge(classification_results1, on=merge_cols, how='left')

#     # Create classification results file for second dataset
#     classification_results2 = df2_with_class[merge_cols + ['classification_result']].dropna()

#     out_path = "/AG/App/FlaskApp/DATA/New_Filtered-Ag/classification-result/llama2-result/" + type + "_final_with_classifications.csv"
#     classification_results2.to_csv(out_path, sep='\t', index=False)

#     # Count results
#     print(type)
#     print("--- Prev count",   len(df1), len(df2), len(classification_results1))
#     count = len(classification_results2)
#     print("New count -- ", count)
    
    
import pandas as pd

types = ["corn", "Ghana", "Liberia", "Mali", "multispectral", "Niger", "Nigeria", "nitrogendilution", "nyield"]

for type in types:
    df1 = pd.read_csv(f'/AG/App/FlaskApp/DATA/Ag-Data/{type}/final_merged_output.csv', sep='\t')
    df2 = pd.read_csv(f'/AG/App/FlaskApp/DATA/New_Filtered-Ag/data/{type}-final.csv', sep='\t')
    classification_results = pd.read_csv(f'/AG/App/FlaskApp/DATA/Ag-Data/{type}/final_with_classifications.csv', sep='\t')
    
    merge_cols = ['title']
                #   ,'abstract' , 'doi', 'authors', 'count']
    common_rows = df1.merge(df2[merge_cols], on=merge_cols, how='inner')
    filtered_classification = classification_results.merge(common_rows[merge_cols], on=merge_cols, how='inner')
    
    out_path = "/AG/App/FlaskApp/DATA/New_Filtered-Ag/classification-result/llama2-result/" + type + "_final_with_classifications.csv"
    filtered_classification.to_csv(out_path, sep='\t', index=False)
    
    print(type)
    print("--- Prev count", len(df1), len(df2), len(classification_results))
    print("New count --", len(filtered_classification))

corn
--- Prev count 105180 63056 105180
New count -- 63056
Ghana
--- Prev count 5732 5527 5732
New count -- 5527
Liberia
--- Prev count 737 685 737
New count -- 685
Mali
--- Prev count 5270 5136 5270
New count -- 5136
multispectral
--- Prev count 98906 77489 98906
New count -- 77489
Niger
--- Prev count 6472 5774 6472
New count -- 5774
Nigeria
--- Prev count 6719 5853 6719
New count -- 5853
nitrogendilution
--- Prev count 128 121 128
New count -- 121
nyield
--- Prev count 5949 4413 5949
New count -- 4413


In [48]:
# Now analysis of this new results


import pandas as pd

# Read the CSV file
ndilution = "/AG/App/FlaskApp/DATA/New_Filtered-Ag/classification-result/llama2-result/nitrogendilution_final_with_classifications.csv"
multispectral = "/AG/App/FlaskApp/DATA/New_Filtered-Ag/classification-result/llama2-result/multispectral_final_with_classifications.csv"
nyield = "/AG/App/FlaskApp/DATA/New_Filtered-Ag/classification-result/llama2-result/nyield_final_with_classifications.csv"
corn = "/AG/App/FlaskApp/DATA/New_Filtered-Ag/classification-result/llama2-result/corn_final_with_classifications.csv"


import pandas as pd

file_path = 'data.xlsx' 
xls = pd.ExcelFile(file_path)

doi_dict = {}
title_dict = {}

for sheet in xls.sheet_names:
    df = pd.read_excel(xls, sheet_name=sheet)
    
    if 'doi' in df.columns:
        doi_dict[sheet] = df['doi'].copy()
    if 'Title' in df.columns:
        title_dict[sheet] = df['Title'].copy()


ndilution_dois = doi_dict.get('id1_references').tolist()
multispectral_doi = doi_dict.get('id2_references').tolist()
nyield_doi = doi_dict.get('id3_references').tolist()
corn_doi = doi_dict.get('id4_references').tolist()
 

ndilution_titles = title_dict.get('id1_references').tolist()
multispectral_titles = title_dict.get('id2_references').tolist()
nyield_titles = title_dict.get('id3_references').tolist()
corn_titles = title_dict.get('id4_references').tolist()
 
 

In [44]:


def do_analysis(multispectral, multispectral_doi):

    df = pd.read_csv(multispectral, sep="\t")

    def find_related_unrelated(text):
        if pd.isna(text):
            return None  # Return None if the text is NaN
        elif 'unrelated' in text.lower():
            return 0
        elif 'related' in text.lower():
            return 1
        else:
            return None  # Return None if neither word is found

    df['Result'] = df.iloc[:, -1].apply(find_related_unrelated)


    df_related = df[df['Result'] == 1]

    article_doi_lower = [doi for doi in multispectral_doi]

    df_related['doi'] = "https://doi.org/" + df_related['doi'].astype(str)
    df_related['doi_lower'] = df_related['doi']

    matching_doi = df_related[df_related['doi_lower'].isin(article_doi_lower)]

    accuracy = len(matching_doi) / len(article_doi_lower) * 100

    # print("------")
    # print(len(df))
    # print(len(df_related))
    # print(len(article_doi_lower))
    # print(f"Accuracy: {accuracy:.2f}%")

    # print("MATCHING SELECTED TITLES ", len(matching_doi))
    # print("---------------")
    
    print("------")
    print(f"Total records: {len(df)}")
    print(f"Related records: {len(df_related)}")
    print(f"Target DOIs: {len(article_doi_lower)}")
    print(f"Matching DOIs found: {len(matching_doi)}")
    print(f"Accuracy: {accuracy:.2f}%")
    print("---------------")


do_analysis(multispectral, multispectral_doi)
do_analysis(corn, corn_doi)
do_analysis(nyield, nyield_doi)
do_analysis(ndilution, ndilution_dois)

/tmp/ipykernel_3703502/3986035725.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['doi'] = "https://doi.org/" + df_related['doi'].astype(str)
/tmp/ipykernel_3703502/3986035725.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['doi_lower'] = df_related['doi']


------
Total records: 77489
Related records: 47666
Target DOIs: 41
Matching DOIs found: 32
Accuracy: 78.05%
---------------
------
Total records: 63056
Related records: 48752
Target DOIs: 21
Matching DOIs found: 12
Accuracy: 57.14%
---------------
------
Total records: 4413
Related records: 3644
Target DOIs: 84
Matching DOIs found: 43
Accuracy: 51.19%
---------------
------
Total records: 121
Related records: 120
Target DOIs: 36
Matching DOIs found: 14
Accuracy: 38.89%
---------------


/tmp/ipykernel_3703502/3986035725.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['doi'] = "https://doi.org/" + df_related['doi'].astype(str)
/tmp/ipykernel_3703502/3986035725.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['doi_lower'] = df_related['doi']
/tmp/ipykernel_3703502/3986035725.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentat

In [59]:
def analyze_classification(csv_file, titles_list, dataset_name):
   
   df = pd.read_csv(csv_file, sep="\t")

   def find_related_unrelated(text):
       if pd.isna(text):
           return None
       elif 'unrelated' in text.lower():
           return 0
       elif 'related' in text.lower():
           return 1
       else:
           return None

   df['Result'] = df.iloc[:, -1].apply(find_related_unrelated)

   df_related = df[df['Result'] == 1]

   article_titles_lower = [title.lower() for title in titles_list]

   df_related['title_lower'] = df_related['title'].str.lower()

   matching_titles = df_related[df_related['title_lower'].isin(article_titles_lower)]

   accuracy = len(matching_titles) / len(article_titles_lower) * 100

   print(f"---{dataset_name.upper()}---")
   print(len(df))
   print(len(df_related))
   print(len(article_titles_lower))
   print(f"Accuracy: {accuracy:.2f}%")

   print("MATCHING SELECTED TITLES ", len(matching_titles))

   df['title_lower'] = df['title'].str.lower()
   df_titles_lower = df['title_lower'].tolist()

   matching_titles_all = [title for title in article_titles_lower if title in df_titles_lower]
   missing_titles = [title for title in article_titles_lower if title not in df_titles_lower]

   matching_count = len(matching_titles_all)
   missing_count = len(missing_titles)

   print("Matching Titles: ", matching_count)
   print("Missing Titles: ", missing_count)
   
   
analyze_classification(multispectral, multispectral_titles, "multispectral")
analyze_classification(corn, corn_titles, "corn")
analyze_classification(nyield, nyield_titles, "nyield")
analyze_classification(ndilution, ndilution_titles, "ndilution")

/tmp/ipykernel_3703502/517613486.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['title_lower'] = df_related['title'].str.lower()


---MULTISPECTRAL---
77489
47666
41
Accuracy: 65.85%
MATCHING SELECTED TITLES  27
Matching Titles:  29
Missing Titles:  12
---CORN---
63056
48752
21
Accuracy: 47.62%
MATCHING SELECTED TITLES  10
Matching Titles:  11
Missing Titles:  10
---NYIELD---
4413
3644
84
Accuracy: 29.76%
MATCHING SELECTED TITLES  25
Matching Titles:  26
Missing Titles:  58
---NDILUTION---
121
120
36
Accuracy: 33.33%
MATCHING SELECTED TITLES  12
Matching Titles:  12
Missing Titles:  24


/tmp/ipykernel_3703502/517613486.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['title_lower'] = df_related['title'].str.lower()
/tmp/ipykernel_3703502/517613486.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['title_lower'] = df_related['title'].str.lower()
/tmp/ipykernel_3703502/517613486.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the document

In [68]:
# Dynamic for DOI 

def do_analysis(multispectral, multispectral_doi, key):
   df = pd.read_csv(multispectral, sep="\t")
   
   def find_related_unrelated(text):
       if pd.isna(text):
           return None
       elif 'unrelated' in text.lower():
           return 0
       elif 'related' in text.lower():
           return 1
       else:
           return None
   
   df['Result'] = df.iloc[:, -1].apply(find_related_unrelated)
   df_related = df[df['Result'] == 1]
   
   article_doi_lower = [doi for doi in multispectral_doi]
   df_related['doi'] = "https://doi.org/" + df_related['doi'].astype(str)
   df_related['doi_lower'] = df_related['doi']
   
   matching_doi = df_related[df_related['doi_lower'].isin(article_doi_lower)]
   
   accuracy = len(matching_doi) / len(article_doi_lower) * 100
   
   print(f"------ {key.upper()} ------")
   print(f"Total records: {len(df)}")
   print(f"Related records: {len(df_related)}")
   print(f"Target DOIs: {len(article_doi_lower)}")
   print(f"Matching DOIs found: {len(matching_doi)}")
   print(f"Accuracy: {accuracy:.2f}%")
   print("MATCHING SELECTED DOIS ", len(matching_doi))
   
   df['doi'] = "https://doi.org/" + df['doi'].astype(str)
   df['doi_lower'] = df['doi']
   df_dois_lower = df['doi_lower'].tolist()
   
   matching_dois_all = [doi for doi in article_doi_lower if doi in df_dois_lower]
   missing_dois = [doi for doi in article_doi_lower if doi not in df_dois_lower]
   
   matching_count = len(matching_dois_all)
   missing_count = len(missing_dois)
   
   print("Matching DOIs: ", matching_count)
   print("Missing DOIs: ", missing_count)
   print("---------------")

# do_analysis(multispectral, multispectral_doi, "multispectral")
# do_analysis(corn, corn_doi, "corn")
do_analysis(nyield, nyield_doi, "nyield")
do_analysis(ndilution, ndilution_dois, "ndilution")



------ NYIELD ------
Total records: 4413
Related records: 3644
Target DOIs: 84
Matching DOIs found: 43
Accuracy: 51.19%
MATCHING SELECTED DOIS  43
Matching DOIs:  47
Missing DOIs:  37
---------------
------ NDILUTION ------
Total records: 121
Related records: 120
Target DOIs: 36
Matching DOIs found: 14
Accuracy: 38.89%
MATCHING SELECTED DOIS  14
Matching DOIs:  14
Missing DOIs:  22
---------------


/tmp/ipykernel_3703502/600873444.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['doi'] = "https://doi.org/" + df_related['doi'].astype(str)
/tmp/ipykernel_3703502/600873444.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['doi_lower'] = df_related['doi']
/tmp/ipykernel_3703502/600873444.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation

In [ ]:
# --- existing code --- ignore 

In [50]:
import pandas as pd

df = pd.read_csv(multispectral, sep="\t")

def find_related_unrelated(text):
    if pd.isna(text):
        return None  # Return None if the text is NaN
    elif 'unrelated' in text.lower():
        return 0
    elif 'related' in text.lower():
        return 1
    else:
        return None  # Return None if neither word is found

df['Result'] = df.iloc[:, -1].apply(find_related_unrelated)


df_related = df[df['Result'] == 1]

article_titles_lower = [title.lower() for title in multispectral_titles]

df_related['title_lower'] = df_related['title'].str.lower()

matching_titles = df_related[df_related['title_lower'].isin(article_titles_lower)]

accuracy = len(matching_titles) / len(article_titles_lower) * 100

print("---MULTISPECTRAL---")
print(len(df))
print(len(df_related))
print(len(article_titles_lower))
print(f"Accuracy: {accuracy:.2f}%")

print("MATCHING SELECTED TITLES ", len(matching_titles))


df['title_lower'] = df['title'].str.lower()
df_titles_lower = df['title_lower'].tolist()  # Titles from the DataFrame

# Find matching titles
matching_titles = [title for title in article_titles_lower if title in df_titles_lower]

# Find missing titles
missing_titles = [title for title in article_titles_lower if title not in df_titles_lower]

# Calculate matching and missing counts
matching_count = len(matching_titles)
missing_count = len(missing_titles)

# Print results
print("Matching Titles: ", matching_count)
print("Missing Titles: ", missing_count)

---MULTISPECTRAL---
77489
47666
41
Accuracy: 65.85%
MATCHING SELECTED TITLES  27
Matching Titles:  29
Missing Titles:  12


/tmp/ipykernel_3703502/3606224568.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['title_lower'] = df_related['title'].str.lower()


In [51]:
df = pd.read_csv(corn, sep="\t")

def find_related_unrelated(text):
    if pd.isna(text):
        return None  # Return None if the text is NaN
    elif 'unrelated' in text.lower():
        return 0
    elif 'related' in text.lower():
        return 1
    else:
        return None  # Return None if neither word is found

df['Result'] = df.iloc[:, -1].apply(find_related_unrelated)


df_related = df[df['Result'] == 1]

article_titles_lower = [title.lower() for title in corn_titles]

df_related['title_lower'] = df_related['title'].str.lower()

matching_titles = df_related[df_related['title_lower'].isin(article_titles_lower)]

accuracy = len(matching_titles) / len(article_titles_lower) * 100

print("---CORN---")
print(len(df))
print(len(df_related))
print(len(article_titles_lower))
print(f"Accuracy: {accuracy:.2f}%")


print("MATCHING SELECTED TITLES ", len(matching_titles))

df['title_lower'] = df['title'].str.lower()
df_titles_lower = df['title_lower'].tolist()  # Titles from the DataFrame

# Find matching titles
matching_titles = [title for title in article_titles_lower if title in df_titles_lower]

# Find missing titles
missing_titles = [title for title in article_titles_lower if title not in df_titles_lower]

# Calculate matching and missing counts
matching_count = len(matching_titles)
missing_count = len(missing_titles)

# Print results
print("Matching Titles: ", matching_count)
print("Missing Titles: ", missing_count)

---CORN---
63056
48752
21
Accuracy: 47.62%
MATCHING SELECTED TITLES  10
Matching Titles:  11
Missing Titles:  10


/tmp/ipykernel_3703502/1470370940.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['title_lower'] = df_related['title'].str.lower()


In [52]:
df = pd.read_csv(nyield, sep="\t")

def find_related_unrelated(text):
    if pd.isna(text):
        return None  # Return None if the text is NaN
    elif 'unrelated' in text.lower():
        return 0
    elif 'related' in text.lower():
        return 1
    else:
        return None  # Return None if neither word is found

df['Result'] = df.iloc[:, -1].apply(find_related_unrelated)


df_related = df[df['Result'] == 1]

article_titles_lower = [title.lower() for title in nyield_titles]

df_related['title_lower'] = df_related['title'].str.lower()

matching_titles = df_related[df_related['title_lower'].isin(article_titles_lower)]

accuracy = len(matching_titles) / len(article_titles_lower) * 100

print("---NITROGEN YIELD---")
print(len(df))
print(len(df_related))
print(len(article_titles_lower))
print(f"Accuracy: {accuracy:.2f}%")

print("MATCHING SELECTED TITLES ", len(matching_titles))

df['title_lower'] = df['title'].str.lower()
df_titles_lower = df['title_lower'].tolist()  # Titles from the DataFrame

# Find matching titles
matching_titles = [title for title in article_titles_lower if title in df_titles_lower]

# Find missing titles
missing_titles = [title for title in article_titles_lower if title not in df_titles_lower]

# Calculate matching and missing counts
matching_count = len(matching_titles)
missing_count = len(missing_titles)

# Print results
print("Matching Titles: ", matching_count)
print("Missing Titles: ", missing_count)

---NITROGEN YIELD---
4413
3644
84
Accuracy: 29.76%
MATCHING SELECTED TITLES  25
Matching Titles:  26
Missing Titles:  58


/tmp/ipykernel_3703502/3426200338.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['title_lower'] = df_related['title'].str.lower()


In [53]:
df = pd.read_csv(ndilution, sep="\t")

def find_related_unrelated(text):
    if pd.isna(text):
        return None  # Return None if the text is NaN
    elif 'unrelated' in text.lower():
        return 0
    elif 'related' in text.lower():
        return 1
    else:
        return None  # Return None if neither word is found

df['Result'] = df.iloc[:, -1].apply(find_related_unrelated)


df_related = df[df['Result'] == 1]

article_titles_lower = [title.lower() for title in ndilution_titles]

df_related['title_lower'] = df_related['title'].str.lower()

matching_titles = df_related[df_related['title_lower'].isin(article_titles_lower)]

accuracy = len(matching_titles) / len(article_titles_lower) * 100

print("---NITROGEN DILUTION---")
print(len(df))
print(len(df_related))
print(len(article_titles_lower))
print(f"Accuracy: {accuracy:.2f}%")

print("MATCHING SELECTED TITLES ", len(matching_titles))


df['title_lower'] = df['title'].str.lower()
df_titles_lower = df['title_lower'].tolist()  # Titles from the DataFrame

# Find matching titles
matching_titles = [title for title in article_titles_lower if title in df_titles_lower]

# Find missing titles
missing_titles = [title for title in article_titles_lower if title not in df_titles_lower]

# Calculate matching and missing counts
matching_count = len(matching_titles)
missing_count = len(missing_titles)

# Print results
print("Matching Titles: ", matching_count)
print("Missing Titles: ", missing_count)

missing_titles

---NITROGEN DILUTION---
121
120
36
Accuracy: 33.33%
MATCHING SELECTED TITLES  12
Matching Titles:  12
Missing Titles:  24


/tmp/ipykernel_3703502/2176313698.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_related['title_lower'] = df_related['title'].str.lower()


['critical n concentration can vary with growth conditions in forage grasses: implications for plant n status assessment and n deficiency diagnosis',
 'nitrogen dilution curves and nitrogen use efficiency during winter-spring growth of annual ryegrass',
 'the efficient use of radiation, water, and nitrogen uptake by low-nitrogen-tolerant broomcorn millet (panicum miliaceum l.) increased grain yield in the loess plateau, china',
 'validation of a critical nitrogen dilution curve for hybrid ryegrass',
 'nitrogen status in maize grown at different row spacings and nitrogen availability',
 'physiological dynamics of maize nitrogen uptake and partitioning in response to plant density and n stress factors: i. vegetative phase',
 'physiological determinants of maize and sunflower grain yield as affected by nitrogen supply',
 'relationship between dynamics of nitrogen uptake and dry matter accumulation in maize crops',
 'the impact of water and nitrogen limitation on maize biomass and resource